# 07 — EDA: pLLM embeddings (TEM-1 / Firnberg)

**Task.** Characterize the ESM **embedding-delta** features before any modeling, so the `08` supervised
gains are read against a known baseline (same discipline as `01`/`05`). These are the features the
supervised arm (arm-3) trains on: per-variant vectors, not the scalar surprisal scores of `05`/`06`.

**Feature blocks** (D035/D036), three per model: `delta_site` (change in the contextual embedding at
the mutated residue — the primary variant-specific feature, D035), `delta_pooled` (whole-sequence mean
delta, D036), `delta_local` (±7-residue window delta, D036).

**What this EDA answers.**
1. Do the deltas exist and are they finite for every variant? (trust-but-verify the Colab output)
2. **Per-block magnitude** — is `delta_site` really larger / more variant-specific than `delta_pooled`?
   That is the empirical justification for D035 being the *primary* feature.
3. **Dimensionality** — PCA scree per model: how many components carry the variance? Sets the
   `08` reduction budget (D037).
4. **Signal before modeling** — how well does a single top principal component separate functional
   from non-functional (point-biserial + single-PC AUC), per block and model?
5. **Redundancy** — how correlated are models to each other in PC space?
6. **Leakage guard** (`CONVENTIONS.md`) — assert no `seq_id`/`position`/label-derived column is in the
   feature matrix.

**Inputs read from disk** (never in-memory state, per CLAUDE.md):
- labels: `data/processed/traditional_ml_aa_identity/modeling_dataset.parquet`
- embeddings: `data/features/plm_embeddings/pllm_embeddings.parquet` — produced by the Colab twin
  `07a_pllm_embedding_extraction_colab.ipynb`; **run that first** and drop the parquet into place. If
  it is absent this notebook prints a PREVIEW banner and stops cleanly.


In [ ]:
# self-contained: resolve project root via .projectroot, read from disk
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root)); from paths import *

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 42
MODEL_ORDER = ["esm1b","esm1v","esm2_150m","esm2_650m","esm2_3b","esmc_300m","esmc_600m"]
MODEL_LABEL = {"esm1b":"ESM-1b 650M","esm1v":"ESM-1v 650M","esm2_150m":"ESM-2 150M",
               "esm2_650m":"ESM-2 650M","esm2_3b":"ESM-2 3B","esmc_300m":"ESM-C 300M","esmc_600m":"ESM-C 600M"}
BLOCKS = ["delta_site","delta_pooled","delta_local"]
BLOCK_LABEL = {"delta_site":"delta @ site (D035)","delta_pooled":"pooled delta (D036)","delta_local":"local-window delta (D036)"}
# project palette: blues, greens, purples, dark pinks only
MODEL_COLORS = {"esm1b":"#2c6fbb","esm1v":"#1f4e8c","esm2_150m":"#57a773","esm2_650m":"#2e8b57",
                "esm2_3b":"#1d6b3f","esmc_300m":"#7a4fa3","esmc_600m":"#b03060"}
BLOCK_COLORS = {"delta_site":"#2c6fbb","delta_pooled":"#2e8b57","delta_local":"#7a4fa3"}
PAL = {"blue":"#2c6fbb","green":"#2e8b57","purple":"#7a4fa3","pink":"#b03060","grey":"#9aa0a6"}
sns.set_theme(style="whitegrid")

NBNAME = "07_EDA_pllm_embeddings"
FIGDIR = FIGURES/NBNAME; FIGDIR.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("root:", root, "| seed:", SEED)


## 1. Load and validate — trust, but verify the Colab output

If the embeddings parquet is missing, this notebook cannot run the EDA. Rather than error, it flips
into **PREVIEW mode**: prints what it would need and stops, so the notebook is reviewable before the
Colab extraction lands.

In [ ]:
df = pd.read_parquet(PROCESSED/"traditional_ml_aa_identity"/"modeling_dataset.parquet")
EMB_PATH = FEATURES_PLM_EMB/"pllm_embeddings.parquet"

PREVIEW = not EMB_PATH.exists()
if PREVIEW:
    print("="*78)
    print("PREVIEW MODE — embeddings not found at:")
    print("   ", EMB_PATH)
    print("Run 1.5 - Colab notebooks/07a_pllm_embedding_extraction_colab.ipynb on a GPU, download")
    print("out/pllm_embeddings.parquet, and drop it at the path above. Then re-run this notebook.")
    print("="*78)
else:
    emb = pd.read_parquet(EMB_PATH)
    # --- boundary checks (CLAUDE.md): trust nothing that crossed from the Colab run ---
    assert df.seq_id.is_unique and emb.seq_id.is_unique, "duplicate seq_id"
    assert set(emb.seq_id) == set(df.seq_id), "embedding/label seq_id sets differ"
    assert len(emb) == len(df), f"row count mismatch {len(emb)} vs {len(df)}"
    data = df.merge(emb, on="seq_id", how="inner", validate="1:1").sort_values(
        ["position","mut_aa"]).reset_index(drop=True)
    y = data.DMS_score_bin.values.astype(int)
    feat_cols = [c for c in emb.columns if c != "seq_id"]
    # which (model, block) pairs actually arrived
    present = [(m,b) for m in MODEL_ORDER for b in BLOCKS
               if any(c.startswith(f"{m}__{b}__") for c in feat_cols)]
    def block_cols(m,b): return [c for c in feat_cols if c.startswith(f"{m}__{b}__")]
    assert present, "no recognized {model}__{block}__ columns present"
    print(f"variants={len(data)} | class balance={np.bincount(y).tolist()} | "
          f"feature cols={len(feat_cols)} | (model,block) present={len(present)}")
    print("present:", [f"{m}/{b}({len(block_cols(m,b))}d)" for m,b in present])


## 2. Data quality — finiteness and leakage guard

Two non-negotiables before any EDA: every embedding value is finite (a NaN means a model/position was
skipped), and no identity/label column leaked into the feature matrix (`CONVENTIONS.md`).

In [ ]:
if not PREVIEW:
    vals = data[feat_cols].values.astype(np.float32)
    n_nan = int(np.isnan(vals).sum())
    print(f"non-finite feature values: {n_nan}  (must be 0)")
    assert n_nan == 0, "non-finite embeddings present"

    # leakage guard: no feature column is an identity/label field or a near-perfect label proxy.
    # chunked float32 per-feature |corr(x,y)| — O(N*D), never builds a D×D matrix.
    forbidden = ("seq_id","position","DMS","score","bin","wt_aa","mut_aa")
    assert not any(any(f in c for f in forbidden) for c in feat_cols), "identity/label column in features"
    yz = (y - y.mean()); yz = yz/(np.linalg.norm(yz)+1e-12)
    mx = 0.0
    for s in range(0, vals.shape[1], 2048):
        blk = vals[:, s:s+2048].copy(); blk -= blk.mean(0)
        nrm = np.linalg.norm(blk, axis=0) + 1e-12
        r = np.abs((blk * yz[:,None]).sum(0) / nrm)
        mx = max(mx, float(np.nanmax(r)))
    print(f"max |corr(feature, y)| over {vals.shape[1]} dims = {mx:.3f}  -> {'OK' if mx<0.99 else 'LEAKAGE'}")
    assert mx < 0.99, "a single embedding dim tracks the label almost perfectly — leakage?"


## 3. Per-block magnitude — is `delta_site` the primary feature?

D035's claim is that the site-level delta carries variant-specific signal a whole-sequence mean washes
out. Test it directly: per variant, the L2 norm of each block. If `delta_site` >> `delta_pooled` in
norm and in across-variant variance, the primary-feature choice is justified by the data, not asserted.

In [ ]:
if not PREVIEW:
    mag_rows=[]
    for m,b in present:
        v = data[block_cols(m,b)].values
        norms = np.linalg.norm(v, axis=1)
        mag_rows.append(dict(model=m, block=b,
            norm_median=float(np.median(norms)), norm_iqr=float(np.subtract(*np.percentile(norms,[75,25]))),
            across_var=float(v.var(0).mean())))   # mean per-dim variance across variants
    mag=pd.DataFrame(mag_rows)
    piv=mag.pivot(index="model",columns="block",values="norm_median").reindex(
        [m for m in MODEL_ORDER if m in mag.model.values])[[b for b in BLOCKS if b in mag.block.values]]
    print("median per-variant L2 norm by block:"); print(piv.round(3).to_string())
    # ratio site/pooled
    if {"delta_site","delta_pooled"} <= set(mag.block):
        rr = (piv["delta_site"]/piv["delta_pooled"]).round(2)
        print("\ndelta_site / delta_pooled norm ratio (>1 supports D035):"); print(rr.to_string())


## 4. Dimensionality — PCA scree per model (sets the `08` reduction budget)

Each block is high-dimensional (640–2560 d). Standardize, then PCA per (model, block) and record how
many components reach 90% / 95% variance. This is what tells `08` a reduction to ~50–200 components is
safe (D037) — most of the variance lives in far fewer dims than the raw width.

In [ ]:
if not PREVIEW:
    pca_rows=[]; scree={}
    for m,b in present:
        X = StandardScaler().fit_transform(data[block_cols(m,b)].values)
        k = min(100, X.shape[1], X.shape[0]-1)
        p = PCA(n_components=k, random_state=SEED).fit(X)
        cum = np.cumsum(p.explained_variance_ratio_)
        n90 = int(np.searchsorted(cum, 0.90)+1); n95=int(np.searchsorted(cum,0.95)+1)
        pca_rows.append(dict(model=m, block=b, dims=X.shape[1],
            pc_for_90=n90, pc_for_95=n95, top10_var=float(cum[min(9,len(cum)-1)])))
        scree[(m,b)] = p.explained_variance_ratio_
    pca_df=pd.DataFrame(pca_rows)
    print("components needed to reach 90% / 95% variance (of first 100 PCs):")
    print(pca_df.to_string(index=False))


## 5. Signal before modeling — how well does one PC separate the classes?

For each (model, block), take the top principal components and measure single-feature association with
the functional label: point-biserial r on PC1, and the best single-PC ROC-AUC among the top 10. This is
the "known baseline" the `08` supervised model must beat — if a single PC already separates well, the
signal is strong and low-dimensional; if not, the classifier has to combine many dims.

In [ ]:
if not PREVIEW:
    assoc_rows=[]
    for m,b in present:
        X = StandardScaler().fit_transform(data[block_cols(m,b)].values)
        k = min(10, X.shape[1], X.shape[0]-1)
        Z = PCA(n_components=k, random_state=SEED).fit_transform(X)
        # point-biserial on PC1
        pb = stats.pointbiserialr(y, Z[:,0]).correlation
        # best single-PC AUC among top-k (orient each PC to y)
        aucs=[max(roc_auc_score(y, Z[:,j]), roc_auc_score(y, -Z[:,j])) for j in range(k)]
        assoc_rows.append(dict(model=m, block=b, pc1_pointbiserial=float(pb),
            best_single_pc_auc=float(np.max(aucs)), best_pc=int(np.argmax(aucs)+1)))
    assoc=pd.DataFrame(assoc_rows).sort_values("best_single_pc_auc", ascending=False)
    print("single-PC association with functional label (top of list = strongest EDA signal):")
    print(assoc.round(3).to_string(index=False))


## 6. Redundancy — how much do models agree in PC space?

If two models' top PCs are highly correlated, they carry redundant signal and stacking all seven buys
little. Correlate PC1 of the primary `delta_site` block across models.

In [ ]:
if not PREVIEW:
    site_models=[m for m,b in present if b=="delta_site"]
    if len(site_models)>=2:
        pc1={}
        for m in site_models:
            X=StandardScaler().fit_transform(data[block_cols(m,"delta_site")].values)
            z=PCA(n_components=1,random_state=SEED).fit_transform(X)[:,0]
            pc1[m]= z if roc_auc_score(y,z)>=roc_auc_score(y,-z) else -z   # orient to y
        C=pd.DataFrame(pc1).corr(method="spearman")
        print("Spearman corr of oriented delta_site PC1 across models:")
        print(C.round(2).to_string())
    else:
        print("fewer than 2 models with delta_site — redundancy view skipped")


## 7. Figures

Per convention: `results/figures/07_EDA_pllm_embeddings/`, generic filenames, palette restricted to
blues / greens / purples / dark pinks.

In [ ]:
if not PREVIEW:
    # 7a. per-block median norm, grouped by model (the D035 magnitude story)
    fig,ax=plt.subplots(figsize=(11,4.6))
    mods=[m for m in MODEL_ORDER if m in mag.model.values]; x=np.arange(len(mods)); w=0.26
    for i,b in enumerate([bb for bb in BLOCKS if bb in mag.block.values]):
        vals=[mag[(mag.model==m)&(mag.block==b)].norm_median.values[0] for m in mods]
        ax.bar(x+i*w, vals, w, color=BLOCK_COLORS[b], label=BLOCK_LABEL[b], edgecolor="black", linewidth=.4)
    ax.set_xticks(x+w); ax.set_xticklabels([MODEL_LABEL[m] for m in mods], rotation=25, ha="right")
    ax.set_ylabel("median per-variant L2 norm"); ax.set_title("Embedding-delta magnitude by block (D035 vs D036)")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(FIGDIR/"block_magnitude.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"block_magnitude.png",bbox_inches="tight",dpi=200); plt.show()

    # 7b. PCA scree — cumulative variance, one line per model (delta_site block)
    fig,ax=plt.subplots(figsize=(8,5))
    for m in [mm for mm in MODEL_ORDER if (mm,"delta_site") in scree]:
        cum=np.cumsum(scree[(m,"delta_site")])
        ax.plot(np.arange(1,len(cum)+1), cum, color=MODEL_COLORS[m], lw=1.8, label=MODEL_LABEL[m])
    ax.axhline(0.9,ls="--",color=PAL["grey"],lw=1); ax.axhline(0.95,ls=":",color=PAL["grey"],lw=1)
    ax.set_xlabel("principal component"); ax.set_ylabel("cumulative variance explained")
    ax.set_title("PCA scree — delta_site block (dashed 90%, dotted 95%)")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(FIGDIR/"pca_scree.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"pca_scree.png",bbox_inches="tight",dpi=200); plt.show()

    # 7c. single-PC AUC heatmap (model x block)
    hm=assoc.pivot(index="model",columns="block",values="best_single_pc_auc").reindex(
        [m for m in MODEL_ORDER if m in assoc.model.values])[[b for b in BLOCKS if b in assoc.block.values]]
    fig,ax=plt.subplots(figsize=(7,4.6))
    sns.heatmap(hm, annot=True, fmt=".3f", cmap="PuBuGn", vmin=0.5, vmax=max(0.6,float(hm.values.max())),
                cbar_kws={"label":"best single-PC ROC-AUC"}, ax=ax)
    ax.set_title("Single-PC separation of functional label"); ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_yticklabels([MODEL_LABEL.get(t.get_text(),t.get_text()) for t in ax.get_yticklabels()], rotation=0)
    fig.tight_layout(); fig.savefig(FIGDIR/"single_pc_auc.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"single_pc_auc.png",bbox_inches="tight",dpi=200); plt.show()


## 8. Save tables

In [ ]:
if not PREVIEW:
    mag.round(4).to_csv(TABLES/f"{NBNAME}_block_magnitude.csv", index=False)
    pca_df.to_csv(TABLES/f"{NBNAME}_pca_variance.csv", index=False)
    assoc.round(4).to_csv(TABLES/f"{NBNAME}_single_pc_association.csv", index=False)
    print("saved tables:")
    for f in sorted(TABLES.glob(f"{NBNAME}_*.csv")): print(" ", f.name)
    print("saved figures:")
    for f in sorted(FIGDIR.glob("*.pdf")): print(" ", f.name)
else:
    print("PREVIEW mode — no tables written. Drop the embeddings parquet in place and re-run.")


## 9. Summary

Read the numbers off the tables above; the interpretation to state next to them:

- **Is `delta_site` the primary feature (D035)?** `block_magnitude` — the site delta should dominate the
  pooled delta in per-variant norm and variance. If it does, the primary-feature choice is data-backed.
- **How compressible is the matrix (D037)?** `pca_variance` — the number of PCs for 90/95% variance is
  the reduction budget `08` uses. Far fewer than the raw width means PCA-in-fold is safe, not lossy.
- **How strong is the signal before training?** `single_pc_association` — the best single-PC AUC per
  block is the pre-modeling floor. `08`'s supervised model combines dims and should beat it.
- **Redundancy across models** — highly correlated PC1s mean the ladder is partly redundant; a subset
  may suffice.

**Limitation, stated where the number lives (D039).** These embeddings are sequence-only, so — like the
zero-shot scores — they are shaped by foldability/stability and can under-represent a catalytic-but-
stable knockout. This EDA measures separability of the *training* signal; whether that signal is the
clinically right one is what the structure features (D038) and the wet-lab panel test.